In [ ]:
import os
import sys
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Create Deformable Linear Object (DLO) environment
The example below demonstrates how to construct a DLO simulation environment and render a 3D frame. The configuration settings are documented in the code.

Try enabling `debug_render=True` and experiment with the number and placement of markers.

In [ ]:
from panda3d.core import loadPrcFileData

loadPrcFileData("", "window-type offscreen")
loadPrcFileData("", "win-size 640 480")

import numpy as np
from PIL import Image
from IPython.display import display

import jax

jax.config.update("jax_enable_x64", True)

from ajx.example_graphics.environment_scene import EnvironmentScene
from ajx.example_graphics.application import Application
from ajx.example_environments.dlo import DLO, DLOSettings, CableParameters
from ajx.simulation import SimulationSettings, Solver
import jax.numpy as jnp
import ajx.math as math
from ajx import Transform


timestep = 1/60
grippermc_to_marker = jnp.array([0.0478024, 0, 0])
environment = DLO(
    sim_settings=SimulationSettings(timestep, True, Solver.DENSE_LINEAR),
    env_settings=DLOSettings.create(
        n_segments=50,
        length=0.6,
        outer_radius=0.015,
        inner_radius=0.013,
        density=1000,
        pose_estimate_linear_offsets=[0.10, 0.20, 0.30, 0.40, 0.50],
        gripper1_offset=Transform(grippermc_to_marker, math.Rotations.identity),
        gripper2_offset=Transform(-grippermc_to_marker, math.Rotations.identity),
        loose_end=False,
    ),
)
environment.camera_rot = math.quat_from_axis_angle(jnp.array([0.0, 0.0, 1.0]), jnp.pi)
nu = 0.333
E = 1e7
cable_param = CableParameters(
    youngs_modulus=E,
    shear_modulus=E / (2 * (1 + nu)),
    damping=environment.default_param.sparse_param.cable_param.damping,
)

env_param = environment.default_param.tree_replace(
    src={"sparse_param.cable_param": cable_param}
)

initial_state = environment.get_neutral_state(env_param)

scene = EnvironmentScene(environment, env_param, initial_state, show_text=False, show_fps=False, debug_render=False)
if "app" in locals():
    app.destroy()
    del app
app = Application(scene, 60, "default", headless=True)
scene.update_geometry()
img = app.get_headless_frame()
display(img)

## Simulate and animate the DLO

The below example shows how to simulate the DLO headless (without interactive 3D grahpics). Each frame is still rendered and an animation is created at the end.

Try experimenting with the control signal $\mathbf u$.

In [ ]:

import imageio
from IPython.display import Video
writer = imageio.get_writer("animation.mp4", fps=60)

# Create an interesting control signal
horizon = 500
u = np.zeros([horizon, 12])
u[:100,0] = -0.2
u[150:250,3] = 1.0
u[150:250,2] = 0.05
u[300:350,5] = 1.0
u[300:400,11] = 0.4

u = jnp.array(u)
store_poses_at = [150, 300]
marker_poses_list = []

env_step = jax.jit(environment.step)
state = initial_state
frames = []

# Simulation loop
for i in range(horizon):
    # Step the environment and store the observation
    state, observations = env_step(state, u[i], env_param)
    scene.state = state
    scene.update_geometry()
    app.graphicsEngine.renderFrame()
    frame = app.get_headless_frame()
    writer.append_data(np.array(frame))
    if i in store_poses_at:
        marker_poses_list.append(observations)
writer.close()

Video("animation.mp4")

## Sim2sim parameter estimation
The example below demonstrates how to estimate the parameters (Young’s modulus and shear modulus) that reproduce the final pose of the previously generated trajectory. The code defines two residual functions, $\mathbf{r}_{\text{fwd}}$ and $\mathbf{r}_{\text{inv}}$, which map a set of parameters $\mathbf{\theta}$ to a vector of residuals.

The objective is to find the parameters that minimize the sum of squared residuals:
$$
\hat{\mathbf \theta} = \text{argmin}_\theta ||r(\mathbf \theta)||^2.
$$
While many efficient solvers exist for least-squares optimization, this example uses a basic [Gauss-Newton](https://en.wikipedia.org/wiki/Gauss%E2%80%93Newton_algorithm) method.

### Forward residual
The forward dynamics residual $\mathbf{r}_{\text{fwd}}$ simulates a single time step of the system and returns the resulting linear and angular velocities at the next time step. Minimizing the squared residuals corresponds to finding a stationary state, assuming the initial state has zero velocity.

### Inverse residual
The inverse dynamics residual $\mathbf{r}_{\text{inv}}$ computes the forces required to drive the system from a given state to a state with zero target velocity. Minimizing the squared residuals likewise corresponds to finding a stationary state, under the assumption that the initial state has zero velocity.

### Which Is Best?
Both residuals enforce static equilibrium, but from different perspectives: the forward residual enforces that the system remains at rest under simulation, while the inverse residual enforces that the net forces acting on the system are zero.
Try running the code and compare the results. Why does one approach converge significantly faster than the other? Try reduce the Gauss-Newton damping to zero.

In [ ]:
from ajx.tree_util import tangent_jacfwd
from ajx.definitions import State, GeneralizedVelocity
from ajx.example_environments.dlo import DLOState
target_pose = state.conf
lock_targets = state.lock_targets
start_err = jnp.linalg.norm(state.gvel.data)
env = environment
n_bodies = env.env_settings.n_segments
multipliers = jnp.zeros_like(state.multipliers)

def forward_dynamics_residual(param):
    gvel = GeneralizedVelocity(jnp.zeros([n_bodies+6, 6]))
    state = DLOState(target_pose, gvel, lock_targets, multipliers)
    next_state = env.step_state(state, jnp.zeros([12]), param)
    return next_state.gvel.data.flatten()

def inverse_dynamics_residual(param):
    gvel = GeneralizedVelocity(jnp.zeros([n_bodies+6, 6]))
    state = DLOState(target_pose, gvel, lock_targets, multipliers)
    force = env.sim.inverse_dynamics(state, gvel, jnp.zeros([12]), param)
    return force.flatten()

residual = inverse_dynamics_residual
jac_r = tangent_jacfwd(residual)

def gauss_newton(x0, n_iter, damping):
    x = x0
    for i in range(n_iter):
        rx = residual(x)
        J = jac_r(x) 

        # Gauss-Newton step: solve (J^T J) delta = -J^T r
        JTJ = J.T @ J + jnp.eye(J.shape[1]) * damping
        JTr = J.T @ rx
        delta = -jnp.linalg.solve(JTJ, JTr)

        x = x.retract(delta)
        print(f"Iter: {i}\t |rx|: {jnp.linalg.norm(rx)}")
    return x, J



# Pretend that stiffness is unknown
guess_param = env.default_param.tree_replace(src={
    "sparse_param.cable_param.youngs_modulus":  1e4,
    "sparse_param.cable_param.shear_modulus":  1e4,
})

# Set stiffness as tunable
guess_param = guess_param.replace(
    tangent_restrictions=(
        "sparse_param.cable_param.youngs_modulus",
        "sparse_param.cable_param.shear_modulus",
    )
)

# Optimize
estimated_param, Js = gauss_newton(guess_param, n_iter=50, damping=0.0)
print(jnp.linalg.cond(Js))


In [ ]:
# Print results
youngs_modulus = estimated_param.sparse_param.cable_param.youngs_modulus
shear_modulus = estimated_param.sparse_param.cable_param.shear_modulus
poissons_ratio = youngs_modulus / (2 * shear_modulus) - 1
print(f"Poisson's ratio: {poissons_ratio}")
print(
    f"Youngs modulus, shear modulus: \t{youngs_modulus}, {shear_modulus}"
)

# Estimate uncertainty
U, S, V = jnp.linalg.svd(Js)

JTJ_inv = V @ jnp.diag((1/(S+1e-16))**2) @ V.T
residual_sample_variance = jnp.linalg.norm(residual(estimated_param))**2 / (residual(estimated_param).shape[0] - env_param.tangent_size())
cov = JTJ_inv * residual_sample_variance
parameter_std = jnp.sqrt(jnp.diag(cov))
print(f"Standard deviation estimate: \t\t{parameter_std[:3]}")

# Reconstructing configuration from marker poses

In the next example, the DLO state is reconstructed (approximated) by interpolating between the marker poses. The interpolation is done in multiple steps
1. The marker poses are converted to transforms for the bodies (segments or grippers) that the markers are attached to
2. The body transforms are converted to transforms for the constraint frames attached to the body. This results in two transforms per segment in the DLO, one for "frame a" and one for "frame b".
3. The "frame a" transform list and "frame b" transform list are interpolated separately. To get the actual segment poses, each body transform is reconstructed twice, once with the "frame a" list and once with the "frame b" list. The final pose is the average between the two. 

See if you can recreate the state corresponding to the other set of marker transforms stored in `marker_poses_list`.

In [ ]:
from ajx.example_environments.locked_dlo import LockedDLO
env = LockedDLO(
    sim_settings=SimulationSettings(timestep, True, Solver.DENSE_LINEAR),
    env_settings=DLOSettings.create(
        n_segments=environment.env_settings.n_segments,
        length=0.6,
        outer_radius=0.015,
        inner_radius=0.013,
        density=1000,
        pose_estimate_linear_offsets=[0.10, 0.20, 0.30, 0.40, 0.50],
        gripper1_offset=Transform(grippermc_to_marker, math.Rotations.identity),
        gripper2_offset=Transform(-grippermc_to_marker, math.Rotations.identity),
        loose_end=False,
    ),
)
env.camera_rot = math.quat_from_axis_angle(jnp.array([0.0, 0.0, 1.0]), jnp.pi)
env_param = env.default_param.tree_replace(
    src={"sparse_param.cable_param": cable_param}
)

n_confs = len(observations)
marker_transforms = Transform(
    observations.reshape(-1,7)[:,:3],
    observations.reshape(-1,7)[:,3:],
)
concat_pose = jnp.stack(observations)

body_transforms = env.convert_marker_transforms_to_body_transforms(
    marker_transforms
)
frame_transforms = env.convert_body_transforms_to_frame_transforms(
    env_param, body_transforms
)
state = env.get_state_by_interface_interpolation(
    env_param, [frame_transforms[0], frame_transforms[1], body_transforms]
)

# Ensure the state is aligned with the marker poses
n_poses = len(env.sim.sensor_list)
observed_marker_poses_flat = env.observe_state(state, jnp.zeros(12), env_param)
observed_marker_poses_arr = observed_marker_poses_flat.reshape(n_poses, 7)
observed_marker_poses = Transform(observed_marker_poses_arr[:,:3], observed_marker_poses_arr[:,3:])
marker_residual = jax.vmap(Transform.log_map)(observed_marker_poses, marker_transforms)
squared_error = jnp.sum(marker_residual**2)
print(f"Marker error: {squared_error}")

scene.replace_environment(env, env_param, state)
scene.update_geometry()
interp_img = app.get_headless_frame()
display(interp_img)

### Reconstruction using optimization
In the next example, the configuration is reconstructed using optimization, **assuming that the parameters are known**. 

In [ ]:
target_pose = state.conf
lock_targets = state.lock_targets
start_err = jnp.linalg.norm(state.gvel.data)
n_bodies = env.env_settings.n_segments
initial_guess_type = "interpolation"

def forward_dynamics_residual(state):
    next_state, observations = env.step(state, jnp.zeros([12]), env_param)
    observation_residual = env.observation_residual(
        concat_pose, observations
    )
    return jnp.concatenate([next_state.gvel.data, observation_residual*1e5], axis=None)

def inverse_dynamics_residual(state):
    gvel = GeneralizedVelocity(jnp.zeros([n_bodies+2, 6]))
    observations = env.observe_state(state, jnp.zeros([12]), env_param)
    observation_residual = env.observation_residual(
        concat_pose, observations
    )
    force = env.sim.inverse_dynamics(state, gvel, jnp.zeros([12]), env_param)
    return jnp.concatenate([force, observation_residual*1e5], axis=None)

residual = forward_dynamics_residual
jac_r = tangent_jacfwd(residual)

def gauss_newton(x0, n_iter, damping):
    x = x0
    for i in range(n_iter):
        rx = residual(x)
        J = jac_r(x) 

        # Gauss-Newton step: solve (J^T J) delta = -J^T r
        JTJ = J.T @ J + jnp.eye(J.shape[1]) * damping
        JTr = J.T @ rx
        delta = -jnp.linalg.solve(JTJ, JTr)

        x = x.retract(delta)
        print(f"Iter: {i}\t |rx|: {jnp.linalg.norm(rx)}")
    return x, J

if initial_guess_type == "interpolation":
    # Option 1: Start from interpolation guess
    state_guess = state.replace(
            tangent_restrictions=("conf",)
    )
elif initial_guess_type == "neutral":
    # Option 2: Start from neutral guess
    state_guess = env.get_neutral_state(env_param).replace(
        tangent_restrictions=("conf", "lock_targets")
    )
elif initial_guess_type == "lock_only":
    # Option 3: Only place lock targets
    state_guess = env.get_neutral_state(env_param).replace(
        tangent_restrictions=("conf",),
        lock_targets=state.lock_targets,
    )
# Optimize
solution, Js = gauss_newton(state_guess, n_iter=10, damping=1e-6)

scene.state = solution
scene.update_geometry()
img = app.get_headless_frame()

display(interp_img)
display(img)
